# Take-home lab: Decisions and experiments

AII 600. **Due Wednesday 11 November 2026, before class. Submit on Canvas.**

**How to submit.** Fill in every exercise cell. Use Kernel, Restart and Run All. When every check prints `ok`, upload the `.ipynb` file on Canvas. Run All stops at the first check that fails: fix that exercise and run all again.

**What the checks print.**

- `ok  name`: the value of `name` matches the reference.
- `not yet`: the value is wrong, or is still `None`. The cell stops with an error. Fix the exercise cell above it.

Each exercise defines the variables it names. Do not rename them. Do not round unless the task says so.

| Ex. | Topic | Defines | Slides |
|---|---|---|---|
| T.1 | Utility: St. Petersburg certainty equivalent | `ce_log` | [Lab 8 slides](https://vsokolov.org/courses/files/600/lab8-decisions.html) |
| T.2 | Utility: St. Petersburg with a finite bankroll | `fair_price_1m` | [Lab 8 slides](https://vsokolov.org/courses/files/600/lab8-decisions.html) |
| T.3 | Kelly: half-Kelly stake | `stake_half` | [Lab 8 slides](https://vsokolov.org/courses/files/600/lab8-decisions.html) |
| T.4 | Winner's curse: break-even bid shading | `k_even` | [Lab 8 slides](https://vsokolov.org/courses/files/600/lab8-decisions.html) |
| T.5 | Value of a test | `prior_lo`, `prior_hi` | [Lab 8 slides](https://vsokolov.org/courses/files/600/lab8-decisions.html) |
| T.6 | Nash: mixed equilibrium | `pA`, `pB` | [Lab 8 slides](https://vsokolov.org/courses/files/600/lab8-decisions.html) |
| T.7 | A/B test: interval for a difference of rates | `diff_lo`, `diff_hi` | none; formula in the task |
| T.8 | A/B test: sample size per arm | `n_arm` | none; formula in the task, [Power and Sample Size Lab](https://vsokolov.org/labs/power-lab.html) |


In [ ]:
from hashlib import sha256

import numpy as np
import pandas as pd
from scipy import stats

def check(name, value, expected, ndigits=6):
    if value is None:
        raise AssertionError(f"not yet: {name} is still None. Fill in the exercise cell above and run it.")
    try:
        x = float(value)
    except (TypeError, ValueError):
        raise AssertionError(f"not yet: {name} should be one number, but it is {type(value).__name__}.")
    if not np.isfinite(x):
        raise AssertionError(f"not yet: {name} is not a finite number ({x}).")
    s = f"{round(x, ndigits):.{ndigits}f}"
    got = sha256(f"aii600-takehome|{name}|{s}".encode()).hexdigest()[:16]
    assert got == expected, f"not yet: {name} = {x:.6g} does not match. Check the formula and the inputs."
    print("ok ", name)


### Exercise T.1: St. Petersburg, log utility

A game pays $2^T$ dollars when the first head is on toss $T$, with chance $2^{-T}$. Utility is $u(x) = \log x$ (natural log).

Inputs: $T = 1, 2, \ldots, 200$.

Formulas:

$$
\text{EU} = \sum_{T=1}^{200} 2^{-T} \log\left(2^{T}\right), \qquad \text{CE} = e^{\text{EU}}.
$$

Define `ce_log`: CE, in dollars. One number, not rounded.


In [ ]:
# T.1: define ce_log
ce_log = None


In [ ]:
check("ce_log", ce_log, "cb9adfc8e2f1cc6c")
print("ok  Exercise T.1")


### Exercise T.2: St. Petersburg with a finite bankroll

The game of T.1 is offered by a casino with a bankroll of \$1,000,000. When $2^T$ would be above \$1,000,000, it pays \$1,000,000.

Inputs: $T = 1, 2, \ldots, 200$.

Formula, the fair price:

$$
\text{price} = \sum_{T=1}^{200} 2^{-T} \min\left(2^{T},\, 1{,}000{,}000\right).
$$

Define `fair_price_1m`: the fair price, in dollars. One number, not rounded.


In [ ]:
# T.2: define fair_price_1m
fair_price_1m = None


In [ ]:
check("fair_price_1m", fair_price_1m, "8276e7b47dea71db")
print("ok  Exercise T.2")


### Exercise T.3: Kelly stake

A bet has payout $O = 5$: a winning \$1 ticket pays back \$5, the \$1 included. Your chance of winning is $c = 0.28$. Your bankroll is $B = \$8{,}000$.

Formulas:

$$
\text{edge} = cO - 1, \qquad f = \frac{cO - 1}{O - 1}, \qquad \text{stake} = \tfrac{1}{2}\, f\, B.
$$

Here $f$ is the Kelly fraction of the bankroll. The stake uses half of it.

Define `stake_half`: the dollar stake at half Kelly. One number, not rounded.


In [ ]:
# T.3: define stake_half
stake_half = None


In [ ]:
check("stake_half", stake_half, "a8423a775fa8d743")
print("ok  Exercise T.3")


### Exercise T.4: winner's curse

Five bidders bid on one item. The cell below simulates 100,000 auctions (seed 5): `value5` is the true value $V$ in each auction, shape `(100000,)`. `signals5` holds the five noisy estimates, shape `(100000, 5)`.

Each bidder bids $k$ times his own estimate. The winner is the bidder with the highest estimate $M$, and he pays his own bid $kM$. His profit in one auction is $V - kM$.

Task: find $k$ at which the average profit over the 100,000 auctions is 0. Solve $\text{mean}(V - kM) = 0$ for $k$.

Define `k_even`: one number, about 0.9, not rounded. The check compares 3 decimals.


In [ ]:
def auctions(n_bidders, n_auctions=100_000, seed=5):
    rng = np.random.default_rng(seed)
    value = rng.normal(100, 10, n_auctions)                                # true value, dollars
    signals = value[:, None] + rng.normal(0, 10, (n_auctions, n_bidders))  # noisy estimates
    return value, signals

value5, signals5 = auctions(5)
print(value5.shape, signals5.shape)


In [ ]:
# T.4: define k_even
k_even = None


In [ ]:
check("k_even", k_even, "e8eb0b91e9884936", ndigits=3)
print("ok  Exercise T.4")


### Exercise T.5: value of a test

A disease has prior probability `prior`. A test has sensitivity 0.95 and specificity 0.99. Utilities, in points: treat gives 90 (healthy or sick). Do not treat gives 100 if healthy and 0 if sick. The cell below defines `priors` (0.000, 0.001, ..., 1.000) and `value_of_test(prior)`: the expected utility of testing and then choosing the better action, minus the best expected utility without the test.

The test costs 1 point: `test_cost = 1`.

Define `prior_lo` and `prior_hi`: the smallest and the largest value in `priors` for which `value_of_test(prior)` is above `test_cost`. Each is one number between 0 and 1.


In [ ]:
sens, spec = 0.95, 0.99
test_cost = 1                                   # points
priors = np.arange(1001) / 1000                 # 0.000, 0.001, ..., 1.000

def eu_best(p_sick):
    return max(90, 100 * (1 - p_sick))          # treat: 90 for sure; do not treat: 100 if healthy

def value_of_test(prior):
    p_pos = sens * prior + (1 - spec) * (1 - prior)
    p_neg = 1 - p_pos
    post_pos = sens * prior / p_pos             # chance of disease after a positive test
    post_neg = (1 - sens) * prior / p_neg       # chance of disease after a negative test
    return p_pos * eu_best(post_pos) + p_neg * eu_best(post_neg) - eu_best(prior)

print(round(value_of_test(0.02), 4))


In [ ]:
# T.5: define prior_lo, prior_hi
prior_lo = None
prior_hi = None


In [ ]:
check("prior_lo", prior_lo, "6f3d0d549e585eec")
check("prior_hi", prior_hi, "8e81cb41bc98c4f4")
print("ok  Exercise T.5")


### Exercise T.6: marble game

Players $A$ and $B$ each show a red or a blue marble at the same time. Payoffs, in dollars, paid by an outside organizer; the other player gets \$0:

| | $B$ red | $B$ blue |
|---|---|---|
| $A$ red | $A$ wins 4 | $B$ wins 2 |
| $A$ blue | $B$ wins 2 | $A$ wins 1 |

Let `pA` be the chance that $A$ shows red and `pB` the chance that $B$ shows red. At the mixed equilibrium each player is indifferent between his two marbles:

- $B$ shows red with chance $p_B$: $A$'s expected payoff from red equals that from blue.
- $A$ shows red with chance $p_A$: $B$'s expected payoff from red equals that from blue.

Define `pA` and `pB`: each one number between 0 and 1, exact, with a fraction written as a decimal (1/8 is 0.125).


In [ ]:
# T.6: define pA, pB
pA = None
pB = None


In [ ]:
check("pA", pA, "1710a6c7acd62fcc")
check("pB", pB, "fc03f3492ae0ca7f")
print("ok  Exercise T.6")


### Exercise T.7: interval for a difference of rates

An A/B test of two banners. Pink: 200 purchases among 500 visitors. Black: 165 purchases among 550 visitors.

Inputs: `n_pink`, `buys_pink`, `n_black`, `buys_black` (cell below).

Formulas, with $\hat p_1 = \text{buys\_pink}/\text{n\_pink}$ and $\hat p_2 = \text{buys\_black}/\text{n\_black}$:

$$
\text{SE} = \sqrt{\frac{\hat p_1 (1 - \hat p_1)}{n_1} + \frac{\hat p_2 (1 - \hat p_2)}{n_2}}, \qquad
(\hat p_1 - \hat p_2) \pm z \cdot \text{SE}, \quad z = 1.96.
$$

Use `stats.norm.ppf(0.975)` for $z$.

Define `diff_lo` and `diff_hi`: the lower and upper ends of the 95% interval for pink minus black. Two numbers, as rates (0.04 means 4 points), not rounded.


In [ ]:
n_pink, buys_pink = 500, 200
n_black, buys_black = 550, 165
print(buys_pink / n_pink, buys_black / n_black)


In [ ]:
# T.7: define diff_lo, diff_hi
diff_lo = None
diff_hi = None


In [ ]:
check("diff_lo", diff_lo, "1c56e9f16a5d0ca3")
check("diff_hi", diff_hi, "24e67267fae872fe")
print("ok  Exercise T.7")


### Exercise T.8: sample size per arm

Task: the number of visitors per arm needed to detect a lift in the purchase rate from $p_{\text{old}} = 0.30$ to $p_{\text{new}} = 0.32$, with 5% significance (two-sided) and 80% power.

Formula:

$$
n = \frac{(z_{0.975} + z_{0.80})^2 \left[p_{\text{old}}(1 - p_{\text{old}}) + p_{\text{new}}(1 - p_{\text{new}})\right]}{(p_{\text{new}} - p_{\text{old}})^2}.
$$

Here $z_{0.975}$ and $z_{0.80}$ are normal quantiles. Use `stats.norm.ppf(0.975)` and `stats.norm.ppf(0.80)`, not 1.96 and 0.84. Round up with `np.ceil` and convert to `int`.

Define `n_arm`: visitors per arm, an integer.


In [ ]:
# T.8: define n_arm
n_arm = None


In [ ]:
check("n_arm", n_arm, "9f17504e476314a4")
print("ok  Exercise T.8")
